In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

In [0]:
import requests
import pandas as pd
import yfinance as yf
from io import StringIO
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType

url = "https://lej2.mooo.com/api/v1/download-shared-object/aHR0cHM6Ly9sZWoxLm1vb28uY29tL3Jhd2RhdGFzZXRzL2Nncy5jc3Y_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1NVkNRQk5BR1dUS1lQN1RZTEhOSCUyRjIwMjYwMjI0JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI2MDIyNFQwNDI2MzRaJlgtQW16LUV4cGlyZXM9NDMyMDAmWC1BbXotU2VjdXJpdHktVG9rZW49ZXlKaGJHY2lPaUpJVXpVeE1pSXNJblI1Y0NJNklrcFhWQ0o5LmV5SmhZMk5sYzNOTFpYa2lPaUpOVmtOUlFrNUJSMWRVUzFsUU4xUlpURWhPU0NJc0ltVjRjQ0k2TVRjM01UazBNREUxT0N3aWNHRnlaVzUwSWpvaVpYVnFhVzRpZlEucFRrLWQ3a1BfYjlmQ0hnZVZRdkJEWXhWUkE3U3lCZGU1Zk0tbW5JVmtpSlhqd2Q0bTZNQk9PU19qb2RBd0thSVB2a19vS0lteDlKdjhlQmtXRFNmQkEmWC1BbXotU2lnbmVkSGVhZGVycz1ob3N0JnZlcnNpb25JZD1udWxsJlgtQW16LVNpZ25hdHVyZT1kNGIyNjk0MjIzMjRjOGU0ODZjZjVlN2U2YmVmOTk1NjQ2YmM3ZTQ2M2JjMTE4NWVlZmIzYWE5MTc0MDI4NGEw"
 
# Fetch the CSV content
response = requests.get(url)

# Decode bytes → string → wrap in StringIO for pandas
pdf = pd.read_csv(StringIO(response.content.decode('utf-8')))

def get_info(symbol):
    try:
        info = yf.Ticker(symbol).info
        return pd.Series({
            "name": info.get("longName", "N/A"),
            "current_price": info.get("currentPrice", info.get("regularMarketPrice", None))
        })
    except:
        return pd.Series({"name": "N/A", "current_price": None})

# Apply row-wise — adds 2 new columns directly
pdf[["name", "current_price"]] = pdf["Symbol"].apply(get_info)

# Convert Pandas → Spark DataFrame
# Recommended: Sanitized column names to avoid Parquet/SparkSQL errors
schema = StructType([
    StructField("Symbol", StringType(), True),
    StructField("Code", DoubleType(), True),
    StructField("LACP", DoubleType(), True),
    StructField("Qty_Hand", IntegerType(), True),         # Replaced .
    StructField("Qty_Avai", IntegerType(), True),         # Replaced .
    StructField("Qty_Q_S", IntegerType(), True),          # Replaced .()
    StructField("Avg_Buy_Prc", DoubleType(), True),       # Replaced .
    StructField("Last", DoubleType(), True),
    StructField("Mkt_Val", DoubleType(), True),           # Replaced .
    StructField("Un_G_L", DoubleType(), True),            # Replaced ./
    StructField("Yr_High", DoubleType(), True),           # Replaced .
    StructField("Yr_Low", DoubleType(), True),            # Replaced .
    StructField("Day_High", DoubleType(), True),          # Replaced .
    StructField("Day_Low", DoubleType(), True),           # Replaced .
    StructField("Vol", LongType(), True),                 # LongType for large volume
    StructField("Lot_Size",  StringType(), True),         # Replaced .
    StructField("Chg",  StringType(), True),
    StructField("Chg_Perc", DoubleType(), True),          # Replaced ()
    StructField("Un_G_L_Perc",  StringType(), True),       # Replaced .()
    StructField("Currency", StringType(), True),
    StructField("Exchg", IntegerType(), True),
    StructField("Qty_Sold", IntegerType(), True),         # Replaced .
    StructField("Qty_Susp", IntegerType(), True),         # Replaced .
    StructField("Bid", DoubleType(), True),
    StructField("Ask", DoubleType(), True),
    StructField("Avg_Sell_Price", DoubleType(), True),     # Replaced .
    StructField("Name", StringType(), True), 
    StructField("Current_Price", DoubleType(), True) 
])

df = spark.createDataFrame(pdf, schema = schema)
display(df)
df.write.mode("overwrite").saveAsTable("workspace.default.stocks")